# CUDA 101: The CUDA Kernel

In [562]:
%%bash
nvidia-smi

Sun Oct 11 03:54:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.57                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:0C:00.0 Off |                  N/A |
|  0%   44C    P8             23W /  350W |    1433MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [563]:
%%bash
for file in *.cpp *.hpp *.cu *.cuh *.so *.py threads __pycache__; do
    rm -rf "$file"
done

In [564]:
%%writefile parse_num.hpp
#pragma once
#include <charconv>
#include <cstdlib>
#include <string_view>
#include <system_error>

template <typename T>
int parse_num(T& val, const std::string_view arg) {
  auto arg_end = arg.data() + arg.size();
  auto [read_end, read_err] = std::from_chars(arg.data(), arg_end, val);
  bool parse_err = read_err != std::errc{};
  bool partial_read = read_end != arg_end;
  if (parse_err || partial_read) {
    return EXIT_FAILURE;
  }
  return EXIT_SUCCESS;
}


Writing parse_num.hpp


In [565]:
%%writefile set_grid_shape.cuh
#pragma once
#include <cstdlib>
#include <cuda_runtime.h>

inline int check_grid_shape(const dim3& grid_shape, const cudaDeviceProp& prop) {
  if (grid_shape.x == 0 || grid_shape.y == 0 || grid_shape.z == 0) {
    return EXIT_FAILURE;
  }
  if (grid_shape.x > prop.maxGridSize[0] || grid_shape.y > prop.maxGridSize[1] || grid_shape.z > prop.maxGridSize[2]) {
    return EXIT_FAILURE;
  }
  return EXIT_SUCCESS;
}

Writing set_grid_shape.cuh


In [566]:
%%writefile set_block_shape.cuh
#pragma once
#include <cstdlib>
#include <cuda_runtime.h>

inline int check_block_shape(const dim3& block_shape, const cudaDeviceProp& prop) {
  if (block_shape.x == 0 || block_shape.y == 0 || block_shape.z == 0) {
    return EXIT_FAILURE;
  }
  auto threads_per_block = block_shape.x * block_shape.y * block_shape.z;
  if (threads_per_block > prop.maxThreadsPerBlock) {
    return EXIT_FAILURE;
  }
  return EXIT_SUCCESS;
}

Writing set_block_shape.cuh


In [567]:
%%writefile threads_host.cuh
#pragma once
#include <iostream>
#include <string>

__host__
void welcome() {
  std::string msg = "This program will launch a GPU kernel and print its thread number";
  std::cout << msg << std::endl;
}

Writing threads_host.cuh


In [568]:
%%writefile threads_device.cuh
#pragma once
#include <cstdint>

__device__
uint32_t get_thread() {
  return blockIdx.x * blockDim.x + threadIdx.x;
}

Writing threads_device.cuh


In [569]:
%%writefile threads_kernel.cuh
#pragma once
#include <cstdint>
#include <cstdio>
#include "threads_device.cuh"

__global__
void kernel() {
  uint32_t thread = get_thread();
  printf("Hello from thread %u!\n", thread);
}

Writing threads_kernel.cuh


In [570]:
%%writefile run_kernel.cuh
#pragma once
#include <cuda_runtime.h>
#include "threads_kernel.cuh"

inline cudaError_t run_kernel(const dim3& grid_shape, const dim3& block_shape) {
  kernel<<<grid_shape, block_shape>>>();
  auto kernel_err = cudaGetLastError();
  if (kernel_err != cudaSuccess) {
    return kernel_err;
  }
  return cudaDeviceSynchronize();
}


Writing run_kernel.cuh


In [ ]:
%%writefile threads.cu
#include <cstdint>
#include <cstdlib>
#include <format>
#include <iostream>
#include <span>
#include <string>
#include "parse_num.hpp"
#include "set_block_shape.cuh"
#include "set_grid_shape.cuh"
#include "run_kernel.cuh"
#include "threads_host.cuh"

int main(int argc, char* argv[]) {
  std::span<char*> args(argv, argc);
  constexpr uint32_t device_id = 0;

  if (args.size() != 3) {
    std::string msg = std::format("Usage: {} <grid size> <block size>", args[0]);
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  auto set_device_err = cudaSetDevice(device_id);
  if (set_device_err != cudaSuccess) {
    std::string msg = std::format("Error setting device: {}", cudaGetErrorString(set_device_err));
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  cudaDeviceProp prop{};
  auto prop_err = cudaGetDeviceProperties(&prop, device_id);
  if (prop_err != cudaSuccess) {
    std::string msg = std::format("Error getting device properties: {}", cudaGetErrorString(prop_err));
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  uint32_t grid_size{};
  auto grid_size_err = parse_num(grid_size, args[1]);
  if (grid_size_err != EXIT_SUCCESS) {
    std::cerr << "Invalid grid size" << std::endl;
    std::exit(EXIT_FAILURE);
  }
  auto grid_shape = dim3{grid_size, 1, 1};
  auto grid_shape_err = check_grid_shape(grid_shape, prop);
  if (grid_shape_err != EXIT_SUCCESS) {
    std::cerr << "Invalid grid shape" << std::endl;
    std::exit(EXIT_FAILURE);
  }

  uint32_t block_size{};
  auto block_size_err = parse_num(block_size, args[2]);
  if (block_size_err != EXIT_SUCCESS) {
    std::cerr << "Invalid block size" << std::endl;
    std::exit(EXIT_FAILURE);
  }
  auto block_shape = dim3{block_size, 1, 1};
  auto block_shape_err = check_block_shape(block_shape, prop);
  if (block_shape_err != EXIT_SUCCESS) {
    std::cerr << "Invalid block shape" << std::endl;
    std::exit(EXIT_FAILURE);
  }

  welcome();
  auto kernel_err = run_kernel(grid_shape, block_shape);
  if (kernel_err != cudaSuccess) {
    std::string msg = std::format("Kernel failed: {}", cudaGetErrorString(kernel_err));
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  return EXIT_SUCCESS;
}


In [572]:
%%bash
nvcc -std=c++20 -arch=native -O3 "threads.cu" -o threads

In [573]:
%%bash
./threads 2 3

This program will launch a GPU kernel and print its thread number
Hello from thread 3!
Hello from thread 4!
Hello from thread 5!
Hello from thread 0!
Hello from thread 1!
Hello from thread 2!


# CUDA 102: Extending Python with C

In [574]:
%%writefile lib_cuda.cuh
#pragma once
#include <cstddef>
#include <cstdint>
#include <cuda_runtime.h>

extern "C" {
  cudaError_t c_set_device(uint32_t device_id);
  void c_welcome();
  size_t c_prop_size();
  cudaError_t c_get_prop(cudaDeviceProp* prop, uint32_t device_id);
  int c_parse_u32(uint32_t* val, const char* str);
  int c_check_grid_shape(dim3 grid_shape, const cudaDeviceProp* prop);
  int c_check_block_shape(dim3 block_shape, const cudaDeviceProp* prop);
  cudaError_t c_run_kernel(dim3 grid_shape, dim3 block_shape);
  const char* c_get_error_string(cudaError_t err);
}


Writing lib_cuda.cuh


In [575]:
%%writefile lib_cuda.cu
#include <cstdint>
#include <string_view>
#include "lib_cuda.cuh"
#include "parse_num.hpp"
#include "run_kernel.cuh"
#include "set_block_shape.cuh"
#include "set_grid_shape.cuh"
#include "threads_host.cuh"

extern "C" {
  cudaError_t c_set_device(uint32_t device_id) {
    return cudaSetDevice(device_id);
  }

  void c_welcome() {
    welcome();
  }

  size_t c_prop_size() {
    return sizeof(cudaDeviceProp);
  }

  cudaError_t c_get_prop(cudaDeviceProp* prop, uint32_t device_id) {
    return cudaGetDeviceProperties(prop, device_id);
  }

  int c_parse_u32(uint32_t* val, const char* str) {
    return parse_num(*val, std::string_view(str));
  }

  int c_check_grid_shape(dim3 grid_shape, const cudaDeviceProp* prop) {
    return check_grid_shape(grid_shape, *prop);
  }

  int c_check_block_shape(dim3 block_shape, const cudaDeviceProp* prop) {
    return check_block_shape(block_shape, *prop);
  }

  cudaError_t c_run_kernel(dim3 grid_shape, dim3 block_shape) {
    return run_kernel(grid_shape, block_shape);
  }

  const char* c_get_error_string(cudaError_t err) {
    return cudaGetErrorString(err);
  }
}


Writing lib_cuda.cu


In [576]:
%%bash
nvcc -std=c++20 -arch=native -O3 --compiler-options '-fPIC' -shared lib_cuda.cu -o lib_cuda.so

In [577]:
%%writefile mod_cuda.py
import ctypes
from ctypes import byref, c_byte, c_char_p, c_int, c_size_t, c_uint32, POINTER, Structure
from pathlib import Path

class Dim3(Structure):
    _fields_ = [
        ("x", c_uint32),
        ("y", c_uint32),
        ("z", c_uint32)
    ]

    def __init__(self, x: int = 1, y: int = 1, z: int = 1):
        super().__init__(x, y, z)

    def __repr__(self) -> str:
        return f"Dim3({self.x}, {self.y}, {self.z})"

try:
    lib = ctypes.CDLL(str(Path(__file__).parent / 'lib_cuda.so'))
except OSError as e:
    raise RuntimeError("Failed to load lib_cuda.so") from e

lib.c_prop_size.argtypes = []
lib.c_prop_size.restype  = c_size_t

class DeviceProp(Structure):
    _fields_ = [
        ("data", c_byte * lib.c_prop_size())
    ]

lib.c_set_device.argtypes = [c_uint32]
lib.c_set_device.restype  = c_int

lib.c_welcome.argtypes = []
lib.c_welcome.restype  = None

lib.c_get_prop.argtypes = [POINTER(DeviceProp), c_uint32]
lib.c_get_prop.restype  = c_int

lib.c_parse_u32.argtypes = [POINTER(c_uint32), c_char_p]
lib.c_parse_u32.restype  = c_int

lib.c_check_grid_shape.argtypes = [Dim3, POINTER(DeviceProp)]
lib.c_check_grid_shape.restype  = c_int

lib.c_check_block_shape.argtypes = [Dim3, POINTER(DeviceProp)]
lib.c_check_block_shape.restype  = c_int

lib.c_run_kernel.argtypes = [Dim3, Dim3]
lib.c_run_kernel.restype  = c_int

lib.c_get_error_string.argtypes = [c_int]
lib.c_get_error_string.restype  = c_char_p

def get_error_string(err: int) -> str:
    return lib.c_get_error_string(err).decode('utf-8')

def set_device(device_id: int) -> None:
    if (err := lib.c_set_device(device_id)) != 0:
        raise RuntimeError(f"Error setting device: {get_error_string(err)}")

def welcome() -> None:
    lib.c_welcome()

def get_prop(device_id: int) -> DeviceProp:
    prop = DeviceProp()
    if (err := lib.c_get_prop(byref(prop), device_id)) != 0:
        raise RuntimeError(f"Error getting device properties: {get_error_string(err)}")
    return prop

def parse_u32(s: str) -> int:
    val = c_uint32()
    if lib.c_parse_u32(byref(val), s.encode('utf-8')) != 0:
        raise ValueError("Failed to parse input as uint32_t")
    return val.value

def check_grid_shape(grid_shape: Dim3, prop: DeviceProp) -> None:
    if lib.c_check_grid_shape(grid_shape, byref(prop)) != 0:
        raise ValueError("Invalid grid shape")

def check_block_shape(block_shape: Dim3, prop: DeviceProp) -> None:
    if lib.c_check_block_shape(block_shape, byref(prop)) != 0:
        raise ValueError("Invalid block shape")

def run_kernel(grid_shape: Dim3, block_shape: Dim3) -> None:
    if (err := lib.c_run_kernel(grid_shape, block_shape)) != 0:
        raise RuntimeError(f"Kernel failed: {get_error_string(err)}")


Writing mod_cuda.py


In [578]:
%%writefile threads.py
import sys
from mod_cuda import (
    check_block_shape,
    check_grid_shape,
    Dim3,
    get_prop,
    parse_u32,
    run_kernel,
    set_device,
    welcome,
)

if __name__ == '__main__':
    device_id = 0

    if len(sys.argv) != 3:
        print(f"Usage: {sys.argv[0]} <grid size> <block size>", file=sys.stderr)
        sys.exit(1)

    try:
        set_device(device_id)
    except RuntimeError as e:
        print(e, file=sys.stderr)
        sys.exit(1)

    try:
        prop = get_prop(device_id)
    except RuntimeError as e:
        print(e, file=sys.stderr)
        sys.exit(1)

    try:
        grid_size = parse_u32(sys.argv[1])
    except ValueError:
        print("Invalid grid size", file=sys.stderr)
        sys.exit(1)
    grid_shape = Dim3(grid_size, 1, 1)
    try:
        check_grid_shape(grid_shape, prop)
    except ValueError:
        print("Invalid grid shape", file=sys.stderr)
        sys.exit(1)

    try:
        block_size = parse_u32(sys.argv[2])
    except ValueError:
        print("Invalid block size", file=sys.stderr)
        sys.exit(1)
    block_shape = Dim3(block_size, 1, 1)
    try:
        check_block_shape(block_shape, prop)
    except ValueError:
        print("Invalid block shape", file=sys.stderr)
        sys.exit(1)

    welcome()
    try:
        run_kernel(grid_shape, block_shape)
    except RuntimeError as e:
        print(e, file=sys.stderr)
        sys.exit(1)


Writing threads.py


In [579]:
%%bash
python3 threads.py 2 3


This program will launch a GPU kernel and print its thread number
Hello from thread 3!
Hello from thread 4!
Hello from thread 5!
Hello from thread 0!
Hello from thread 1!
Hello from thread 2!
